# 04. Neural network

The policy/value network for AlphaZero-style guidance. It is the optional `[nn]`
extra (`pip install -e .[nn]`), which pulls in `torch`. Importing `alpha_rule.nn`
stays torch-free until you touch a torch symbol, so the grammar and MCTS run
without it.

Pieces:

* `GrammarTokenizer` turns a rule string into a fixed-length token-id tensor, from
  a vocabulary built out of the grammar plus four special tokens.
* `FormulaEncoder` is a small Transformer that pools the token sequence into one
  vector.
* `PolicyHead` and `ValueHead` read that vector: policy logits over the vocabulary,
  and a scalar value in `(-1, +1)`.
* `AllenFormulaNet` ties the tokenizer, encoder, and heads together.
* `collate`, `train_step`, and `default_optimizer` turn replay-buffer rows into one
  gradient update.

In component 05 a `NeuralEvaluator` wraps this network as an MCTS `Evaluator`,
feeding priors and a value into `run_self_play`.

In [1]:
import alpha_rule
import torch
print("alpha_rule", alpha_rule.__version__, "| torch", torch.__version__)

alpha_rule 0.1.0 | torch 2.11.0+cu128


## Elements

`alpha_rule.nn`
* `GrammarTokenizer(grammar)`: `vocab`, `encode(rule_string, max_len)`,
  `decode(ids)`. Special tokens: `<PAD>`, `<BOS>`, `<EOS>`, `<END>`. `encode` raises
  if a rule needs more than `max_len` tokens (no silent truncation).
* `FormulaEncoder(vocab_size, d_model, ...)`: token + position embeddings, a
  `TransformerEncoder`, and a CLS-style pooler. PAD positions are masked, so a rule
  encodes the same no matter how much the batch is padded. Returns one
  `(B, d_model)` vector.
* `PolicyHead(d_model, num_productions)` / `ValueHead(d_model)`: the two heads.
* `AllenFormulaNet(tokenizer, ...)`: tokenizer + encoder + heads. `forward(ids)`
  returns `(policy_logits, value)`; `predict(ids)` is the gradient-free inference
  variant used to score search nodes.
* `collate`, `train_step`, and `default_optimizer(model)` (Adam with the AlphaZero
  L2 weight decay): one supervised update from replay rows.

Importing the package is lazy: `from alpha_rule.nn import GrammarTokenizer` does not
import torch; the encoder / model / training symbols resolve torch on first use.

## Tokenizer

The vocabulary is the four special tokens followed by the grammar's own tokens.
`encode` wraps the sequence in `<BOS> ... <EOS>` and pads to `max_len`; the empty
`<ROOT>` sentinel encodes to no tokens. With the matrix wildcard `#` no longer
appearing in rule names, it is no longer a special token, so a custom grammar is
free to use it. If a rule needs more than `max_len` tokens, `encode` raises rather
than truncating (which would silently drop the `<EOS>` / `<END>` tail), so size
`max_len` to your largest rule.

In [2]:
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.nn import GrammarTokenizer

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
tok = GrammarTokenizer(g)
print("vocab size :", tok.vocab_size())
print("vocab      :", tok.vocab)
print("encode 'A B <' (max_len=8) :", tok.encode("A B <", max_len=8).tolist())
print("decode round-trip          :", repr(tok.decode(tok.encode("A B <", max_len=8))))
print("encode '<ROOT>' is BOS+EOS+PAD:", tok.encode("<ROOT>", max_len=5).tolist())

# Over-long rules raise instead of dropping the tail.
try:
    tok.encode("A B < > <", max_len=4)
except ValueError as e:
    print("over-long encode raises:", e)

vocab size : 9
vocab      : ['<PAD>', '<BOS>', '<EOS>', '<END>', 'A', 'B', '<', '>', 'END_RULE']
encode 'A B <' (max_len=8) : [1, 4, 5, 6, 2, 0, 0, 0]
decode round-trip          : 'A B <'
encode '<ROOT>' is BOS+EOS+PAD: [1, 2, 0, 0, 0]
over-long encode raises: Rule needs 7 tokens (BOS + 5 + EOS) but max_len=4. Increase max_len to fit your largest rule.


## The network

`AllenFormulaNet` holds the tokenizer and maps a batch of token-id rows to policy
logits over the full vocabulary and a value in `(-1, +1)` (the tanh head). PAD
positions are masked inside the encoder, so a rule's encoding does not depend on the
batch's padding. The logits are unmasked; component 05's `NeuralEvaluator` masks the
non-applicable productions to `-inf` before the softmax.

In [3]:
from alpha_rule.nn import AllenFormulaNet

torch.manual_seed(0)
model = AllenFormulaNet(tok, d_model=16, nhead=2, num_layers=1, max_len=12)
ids = torch.stack([tok.encode("A", max_len=12), tok.encode("A B <", max_len=12)])
policy_logits, value = model(ids)
print("input ids shape    :", tuple(ids.shape))
print("policy logits shape:", tuple(policy_logits.shape), "(batch, vocab)")
print("value shape        :", tuple(value.shape), "values:", [round(v, 3) for v in value.tolist()])
print("value in (-1, 1)   :", bool((value.abs() < 1).all()))
print("parameters         :", sum(p.numel() for p in model.parameters()))

input ids shape    : (2, 12)
policy logits shape: (2, 9) (batch, vocab)
value shape        : (2,) values: [0.123, 0.093]
value in (-1, 1)   : True
parameters         : 6170


## One training step

`collate` packs replay rows into tensors and `train_step` does one gradient update.
The loss is `MSE(value, z) + cross_entropy(policy, visit_pi)`; `default_optimizer`
wraps Adam with the AlphaZero L2 weight decay. Here is a tiny fixed batch run for a
few steps; the loss should fall (a wiring smoke check, not convergence).

In [4]:
from alpha_rule.nn import train_step, default_optimizer

torch.manual_seed(7)
model = AllenFormulaNet(tok, d_model=16, nhead=2, num_layers=1, max_len=12)
optim = default_optimizer(model, lr=1e-2)

batch = [
    ("A",      {"A": 1.0},           0.5),
    ("A B <",  {"<": 0.7, "B": 0.3}, 1.0),
    ("B",      {"END_RULE": 1.0},   -1.0),
]
first = train_step(model, optim, batch, max_len=12)
for _ in range(20):
    last = train_step(model, optim, batch, max_len=12)
print(f"total loss before: {first.total:.3f}  after: {last.total:.3f}")
print(f"  policy         : {first.policy:.3f}  after: {last.policy:.3f}")
print(f"  value          : {first.value:.3f}  after: {last.value:.3f}")
print("total loss decreased:", last.total < first.total)

total loss before: 3.063  after: 1.225
  policy         : 2.242  after: 0.848
  value          : 0.821  after: 0.376
total loss decreased: True


## Masked policy loss

Replay rows from `run_self_play` also carry the productions that were legal at each
state (the 4th tuple element). `train_step` then masks the non-applicable logits to
`-inf` before the softmax, so the training-time distribution matches what the
`NeuralEvaluator` does at inference.

In [5]:
# A 4-tuple row carries the applicable actions; the loss masks the rest.
masked_batch = [
    ("A", {"A": 1.0}, 0.5, ("END_RULE", "A", "B")),
]
log = train_step(model, optim, masked_batch, max_len=12)
print("masked step total loss:", round(log.total, 4))
print("only END_RULE, A, B contribute to the softmax denominator")

masked step total loss: 1.3491
only END_RULE, A, B contribute to the softmax denominator


## A tiny learnable dataset

A small end-to-end check that the network can actually learn something, not just run
the plumbing. The dataset encodes one interpretable pattern: rules built from `A`
are "good" (value near `+1`) and take `<` after the first event, while rules built
from `B` are "bad" (value near `-1`) and take `>`. We overfit on four rows with
`default_optimizer`, then use `model.predict` (gradient-free inference) to confirm
the trained network recovered both the value split and the policy preference.

In [6]:
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.nn import GrammarTokenizer, AllenFormulaNet, train_step, default_optimizer

torch.manual_seed(0)
tok = GrammarTokenizer(AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">")))
model = AllenFormulaNet(tok, d_model=24, nhead=2, num_layers=1, max_len=12)
optim = default_optimizer(model, lr=5e-3)

# (rule string, policy target, value target)
dataset = [
    ("A",     {"<": 1.0},  0.9),     # A is good; after one A, the move is '<'
    ("A A <", {"A": 1.0},  0.9),
    ("B",     {">": 1.0}, -0.9),     # B is bad; after one B, the move is '>'
    ("B B >", {"B": 1.0}, -0.9),
]
for _ in range(400):
    last = train_step(model, optim, dataset, max_len=12)

def predict(rule):
    ids = tok.encode(rule, max_len=12).unsqueeze(0)
    logits, value = model.predict(ids)        # gradient-free inference
    return float(value.item()), tok.token_of[int(logits[0].argmax())]

print(f"final training loss: {last.total:.3f}\n")
for rule in ("A", "B", "A A <", "B B >"):
    v, top = predict(rule)
    print(f"  {rule!r:8} value={v:+.2f}  top policy={top!r}")

v_A, top_A = predict("A")
v_B, top_B = predict("B")
assert v_A > 0 > v_B, (v_A, v_B)                       # learned the good/bad split
assert top_A == "<" and top_B == ">", (top_A, top_B)   # learned the policy
print("\nexpected: A is good and picks '<', B is bad and picks '>'  (ok)")

final training loss: 0.002

  'A'      value=+0.90  top policy='<'
  'B'      value=-0.90  top policy='>'
  'A A <'  value=+0.90  top policy='A'
  'B B >'  value=-0.90  top policy='B'

expected: A is good and picks '<', B is bad and picks '>'  (ok)


## Basic checks

Asserts mirroring the unit tests. `ok` means the pieces wire up.

In [7]:
import torch
from alpha_rule.grammar.allen import AllenIntervalGrammar
from alpha_rule.nn import GrammarTokenizer, AllenFormulaNet, train_step

g = AllenIntervalGrammar(event_types=("A", "B"), relations=("<", ">"))
tok = GrammarTokenizer(g)

# Vocab: 4 specials + grammar vocab; '#' is no longer a special token.
assert tok.vocab_size() == 4 + len(g.vocab())
assert "#" not in tok.vocab
assert tok.vocab[:4] == ["<PAD>", "<BOS>", "<EOS>", "<END>"]

# encode / decode round-trips, strips <ROOT>, and raises on overflow.
assert tok.decode(tok.encode("A B <", max_len=10)) == "A B <"
assert tok.encode("<ROOT>", max_len=4).tolist()[0] == tok.bos_id
try:
    tok.encode("A B < > <", max_len=4)
    raise AssertionError("expected ValueError")
except ValueError:
    pass

# Forward pass shapes; the value is bounded by tanh.
torch.manual_seed(0)
model = AllenFormulaNet(tok, d_model=16, nhead=2, num_layers=1, max_len=10)
ids = torch.stack([tok.encode("A", max_len=10), tok.encode("A B <", max_len=10)])
logits, value = model(ids)
assert logits.shape == (2, tok.vocab_size())
assert value.shape == (2,) and bool((value.abs() < 1).all())

# predict() matches forward() in eval mode and needs no grad.
model.eval()
with torch.no_grad():
    f_logits, f_value = model(ids)
p_logits, p_value = model.predict(ids)
assert torch.allclose(p_logits, f_logits) and torch.allclose(p_value, f_value)

# A training step returns finite, non-negative loss components.
optim = torch.optim.Adam(model.parameters(), lr=1e-2)
log = train_step(model, optim, [("A", {"A": 1.0}, 0.5)], max_len=10)
assert log.total >= 0 and log.policy >= 0 and log.value >= 0

print("ok")

ok


## Full unit tests

```
python -m alpha_rule.tests.run_tests test_tokenizer
python -m alpha_rule.tests.run_tests test_model
python -m alpha_rule.tests.run_tests test_training_step
```

These need the `[nn]` extra (`torch`). Without it, `run_tests` skips the modules
rather than failing.